In [3]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd

# --------------------------
# 1. 数据准备
# --------------------------
# 使用 Plotly 内置的 gapminder 数据集作为基础（包含国家代码和名称）
# 我们只需要地理信息，所以只取其中一年的数据即可
df = px.data.gapminder().query("year == 2007")

# 创建一个新的分类列，用于给地图上色
# 逻辑：如果是中国(China)，标记为 'China'，否则标记为 'World'
df['highlight'] = df['country'].apply(lambda x: 'China (中国)' if x == 'China' else 'Rest of World')

# --------------------------
# 2. 创建 3D 地球仪
# --------------------------
fig = px.choropleth(
    df,
    locations="iso_alpha",     # 使用 ISO 国家代码定位
    color="highlight",         # 根据我们刚才创建的 highlight 列来上色
    hover_name="country",      # 鼠标悬停显示国家名
    projection="orthographic", # 【关键】设置为 'orthographic' 即为球体投影
    
    # 自定义颜色映射：中国用红色，其他地区用浅灰色（模拟大陆效果）
    color_discrete_map={
        'China (中国)': '#FF0000',      # 红色
        'Rest of World': '#DDDDDD'     # 浅灰色
    }
)

# --------------------------
# 3. 美化与样式调整
# --------------------------
fig.update_geos(
    visible=True,           # 显示地理图层
    showcountries=False,    # 关闭默认的国家边界线（为了视觉上融合成大陆）
    showcoastlines=True,    # 显示海岸线（区分陆地和海洋）
    coastlinecolor="#888888", # 海岸线颜色
    showland=True,          # 显示陆地
    landcolor="#DDDDDD",    # 陆地底色
    showocean=True,         # 显示海洋
    oceancolor="#A4D3EE",   # 海洋颜色（浅蓝）
    projection_rotation=dict(lon=105, lat=35, roll=0) # 初始视角：聚焦亚洲/中国
)

fig.update_layout(
    title_text="3D Globe - Highlighting China",
    margin={"r":0,"t":50,"l":0,"b":0}, # 移除多余边距
    showlegend=True                    # 显示图例
)

# --------------------------
# 4. 在独立窗口打开
# --------------------------
# renderer="browser" 会强制调用系统默认浏览器打开一个新标签页/窗口展示图片
# 这满足了你“独立窗口打开”的需求
fig.show(renderer="browser")

In [4]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np # 我们需要 numpy 来生成一些轨道数据

# ==========================================
# 1. 基础数据准备 (同上一版)
# ==========================================
# 获取基础地理数据
df = px.data.gapminder().query("year == 2007")
# 设置高亮逻辑
df['highlight'] = df['country'].apply(lambda x: 'China (中国)' if x == 'China' else 'Rest of World')

# ==========================================
# 2. 创建基础 3D 地球仪 (底图)
# ==========================================
fig = px.choropleth(
    df,
    locations="iso_alpha",
    color="highlight",
    hover_name="country",
    projection="orthographic", # 3D球体投影
    # 自定义颜色映射
    color_discrete_map={
        'China (中国)': '#FF0000',      # 中国红
        'Rest of World': '#D0D0D0'     # 其他地区浅灰
    }
)

# ==========================================
# 3. 添加卫星轨道和卫星 (新功能)
# ==========================================

# --- 3.1 生成轨道数据 ---
# 生成一条倾斜的轨道路径数据
# 经度从 -180 到 180 绕一圈
orbit_lon = np.linspace(-180, 180, 360)
# 纬度按正弦波变化，模拟倾斜轨道，乘以 75 让它覆盖南北纬 75 度
orbit_lat = np.sin(np.deg2rad(orbit_lon * 2)) * 75

# --- 3.2 添加轨道线图层 ---
fig.add_trace(go.Scattergeo(
    lon=orbit_lon,
    lat=orbit_lat,
    mode='lines', # 模式为线条
    line=dict(
        width=2,
        color='#00FFFF', # 青色发光效果
        dash='dot'       # 虚线，更有科技感
    ),
    name='Satellite Orbit (轨道)', # 图例名称
    hoverinfo='none' # 鼠标悬停不显示信息
))

# --- 3.3 添加卫星点图层 ---
# 我们在轨道上选取几个点作为卫星的位置
sat_indices = [45, 135, 225, 315] # 在轨道数组中选几个索引
sat_lon = orbit_lon[sat_indices]
sat_lat = orbit_lat[sat_indices]

fig.add_trace(go.Scattergeo(
    lon=sat_lon,
    lat=sat_lat,
    mode='markers', # 模式为点
    marker=dict(
        size=12,               # 卫星大小
        color='#FFFF00',       # 亮黄色
        symbol='diamond-wide', # 宽菱形，看起来像人造飞行器
        line=dict(width=2, color='#FFFFFF'), # 白色描边增加立体感
        opacity=0.9
    ),
    name='Satellites (卫星)',
    hoverinfo='lon+lat' # 鼠标悬停显示经纬度
))


# ==========================================
# 4. 美化与样式调整 (全局设置)
# ==========================================
fig.update_geos(
    visible=True,
    showcountries=False,    # 隐藏国界
    showcoastlines=True,    # 显示海岸线
    coastlinecolor="#666666", # 深灰色海岸线
    showland=True,
    landcolor="#D0D0D0",    # 陆地统一底色
    showocean=True,
    oceancolor="#001f3f",   # 【修改】深蓝色海洋，更有太空深邃感
    projection_rotation=dict(lon=110, lat=30, roll=0), # 视角聚焦中国上方
    bgcolor="#000000"       # 【修改】背景改为黑色，模拟太空
)

fig.update_layout(
    title_text="3D Geospatial Visualization with Satellites",
    title_font_color="#FFFFFF", # 标题白色
    margin={"r":0,"t":50,"l":0,"b":0},
    paper_bgcolor="#000000", # 画布背景黑色
    legend=dict(
        font=dict(color="#FFFFFF"), # 图例文字白色
        bgcolor="rgba(0,0,0,0.5)"   # 图例背景半透明黑
    )
)

# ==========================================
# 5. 在独立窗口打开
# ==========================================
# 记得先在 notebook 里运行 !pip install pandas plotly 确保安装
fig.show(renderer="browser")

In [5]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np

# ==========================================
# 1. 准备国家数据
# ==========================================
df = px.data.gapminder().query("year == 2007")

# 定义我们要着重标注的“大国”列表
# 你可以在这个列表中添加任何你想高亮的国家英文名
target_countries = [
    'China', 'United States', 'Russia', 'France', 
    'India', 'Brazil', 'Australia', 'United Kingdom', 
    'Egypt', 'Canada'
]

def set_color_group(country):
    if country == 'China':
        return 'China (中国)'  # 中国依然最显眼
    elif country in target_countries:
        return country       # 其他大国显示原名
    else:
        return 'Others'      # 其他国家归为一类

df['color_group'] = df['country'].apply(set_color_group)

# ==========================================
# 2. 准备文化地标数据 (手动录入经纬度)
# ==========================================
# 这里我为你挑选了一些全球著名的地标
landmarks_data = {
    'Name': [
        '长城', '故宫',              # 中国
        '埃菲尔铁塔',                # 法国
        '克里姆林宫',                # 俄罗斯
        '自由女神像',                # 美国
        '泰姬陵',                    # 印度
        '金字塔',                    # 埃及
        '悉尼歌剧院',                # 澳大利亚
        '救世基督像',                # 巴西
        '大本钟'                     # 英国
    ],
    'Lat': [
        40.4319, 39.9163,           # 中国坐标
        48.8584,                    # 法国坐标
        55.7520,                    # 俄罗斯坐标
        40.6892,                    # 美国坐标
        27.1751,                    # 印度坐标
        29.9792,                    # 埃及坐标
        -33.8568,                   # 澳洲坐标
        -22.9519,                   # 巴西坐标
        51.5007                     # 英国坐标
    ],
    'Lon': [
        116.5704, 116.3971,         # 中国坐标
        2.2945,                     # 法国坐标
        37.6173,                    # 俄罗斯坐标
        -74.0445,                   # 美国坐标
        78.0421,                    # 印度坐标
        31.1342,                    # 埃及坐标
        151.2153,                   # 澳洲坐标
        -43.2105,                   # 巴西坐标
        -0.1246                     # 英国坐标
    ]
}
df_landmarks = pd.DataFrame(landmarks_data)

# ==========================================
# 3. 绘制 3D 地球底图 (国家层)
# ==========================================
fig = px.choropleth(
    df,
    locations="iso_alpha",
    color="color_group",      # 按组上色
    hover_name="country",
    projection="orthographic",
    # 自定义颜色逻辑
    color_discrete_map={
        'China (中国)': '#FF0000',    # 中国红
        'Others': '#DDDDDD',         # 其他小国浅灰
        'Russia': '#1f77b4',         # 俄罗斯蓝
        'United States': '#2ca02c',  # 美国绿
        'France': '#9467bd',         # 法国紫
        # 其他未指定的国家会自动分配默认颜色
    }
)

# ==========================================
# 4. 添加文化地标层 (Marker + Text)
# ==========================================
fig.add_trace(go.Scattergeo(
    lon=df_landmarks['Lon'],
    lat=df_landmarks['Lat'],
    text=df_landmarks['Name'],
    mode='markers+text',     # 同时显示点和文字
    marker=dict(
        size=8,
        color='gold',        # 地标点用金色
        symbol='star',       # 形状用五角星
        line=dict(width=1, color='black') # 黑色描边
    ),
    textposition="top center", # 文字显示在点上方
    textfont=dict(
        size=12,
        color="black"        # 地标文字颜色，选醒目的颜色
    ),
    name='Cultural Landmarks'
))

# ==========================================
# 5. 保留卫星轨道特效 (背景点缀)
# ==========================================
orbit_lon = np.linspace(-180, 180, 360)
orbit_lat = np.sin(np.deg2rad(orbit_lon * 2)) * 80 # 稍微加大倾角覆盖更多区域

fig.add_trace(go.Scattergeo(
    lon=orbit_lon, lat=orbit_lat,
    mode='lines',
    line=dict(width=1, color='#00FFFF', dash='dot'),
    name='Orbit', hoverinfo='none'
))

# ==========================================
# 6. 全局美化
# ==========================================
fig.update_geos(
    visible=True,
    showcountries=True,      # 【开启】显示所有国家的边界
    countrycolor="white",    # 国界线用白色
    showcoastlines=True,
    coastlinecolor="#888888",
    showland=True,
    landcolor="#E5E5E5",
    showocean=True,
    oceancolor="#A4D3EE",    # 恢复清爽的浅蓝海洋，方便看清地标文字
    projection_rotation=dict(lon=105, lat=30, roll=0)
)

fig.update_layout(
    title_text="3D Globe: Major Countries & Cultural Landmarks",
    margin={"r":0,"t":30,"l":0,"b":0},
    showlegend=True,
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01
    )
)

fig.show(renderer="browser")

In [6]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd

# ==========================================
# 1. 数据准备：全球 Top 15 面积大国
# ==========================================
# 数据来源：大致参考世界银行数据 (单位：万平方公里)
# 我们需要手动定义中心点(Lat/Lon)来放置文字标签，防止文字跑偏
top15_data = [
    {'iso': 'RUS', 'name': '俄罗斯', 'rank': 1, 'area': '1,709', 'lat': 60, 'lon': 100},
    {'iso': 'CAN', 'name': '加拿大', 'rank': 2, 'area': '998', 'lat': 60, 'lon': -110},
    {'iso': 'CHN', 'name': '中国', 'rank': 3, 'area': '960', 'lat': 35, 'lon': 105},
    {'iso': 'USA', 'name': '美国', 'rank': 4, 'area': '937', 'lat': 40, 'lon': -100},
    {'iso': 'BRA', 'name': '巴西', 'rank': 5, 'area': '851', 'lat': -10, 'lon': -55},
    {'iso': 'AUS', 'name': '澳大利亚', 'rank': 6, 'area': '769', 'lat': -25, 'lon': 135},
    {'iso': 'IND', 'name': '印度', 'rank': 7, 'area': '328', 'lat': 22, 'lon': 79},
    {'iso': 'ARG', 'name': '阿根廷', 'rank': 8, 'area': '278', 'lat': -35, 'lon': -65},
    {'iso': 'KAZ', 'name': '哈萨克斯坦', 'rank': 9, 'area': '272', 'lat': 48, 'lon': 68},
    {'iso': 'DZA', 'name': '阿尔及利亚', 'rank': 10, 'area': '238', 'lat': 28, 'lon': 3},
    {'iso': 'COD', 'name': '刚果(金)', 'rank': 11, 'area': '234', 'lat': -2, 'lon': 23},
    {'iso': 'GRL', 'name': '格陵兰(丹)', 'rank': 12, 'area': '216', 'lat': 72, 'lon': -40}, # 地理上很大
    {'iso': 'SAU', 'name': '沙特阿拉伯', 'rank': 13, 'area': '215', 'lat': 24, 'lon': 45},
    {'iso': 'MEX', 'name': '墨西哥', 'rank': 14, 'area': '196', 'lat': 23, 'lon': -102},
    {'iso': 'IDN', 'name': '印度尼西亚', 'rank': 15, 'area': '190', 'lat': -2, 'lon': 120},
]
df_top15 = pd.DataFrame(top15_data)

# 生成显示的标签文本，例如: "#1\n1709万"
df_top15['label'] = df_top15.apply(lambda x: f"#{x['rank']} {x['name']}<br>{x['area']}万k㎡", axis=1)

# ==========================================
# 2. 数据准备：50+ 全球文化地标
# ==========================================
landmarks_list = [
    # --- 亚洲 ---
    {'name': '长城', 'lat': 40.43, 'lon': 116.57},
    {'name': '故宫', 'lat': 39.91, 'lon': 116.39},
    {'name': '兵马俑', 'lat': 34.38, 'lon': 109.27},
    {'name': '布达拉宫', 'lat': 29.65, 'lon': 91.11},
    {'name': '泰姬陵', 'lat': 27.17, 'lon': 78.04}, # 印度
    {'name': '吴哥窟', 'lat': 13.41, 'lon': 103.86}, # 柬埔寨
    {'name': '富士山', 'lat': 35.36, 'lon': 138.72}, # 日本
    {'name': '京都金阁寺', 'lat': 35.03, 'lon': 135.72}, # 日本
    {'name': '佩特拉古城', 'lat': 30.32, 'lon': 35.44}, # 约旦
    {'name': '哈利法塔', 'lat': 25.19, 'lon': 55.27}, # 迪拜
    {'name': '婆罗浮屠', 'lat': -7.60, 'lon': 110.20}, # 印尼
    {'name': '仰光大金塔', 'lat': 16.79, 'lon': 96.14}, # 缅甸
    {'name': '麦加禁寺', 'lat': 21.42, 'lon': 39.82}, # 沙特
    {'name': '波斯波利斯', 'lat': 29.93, 'lon': 52.89}, # 伊朗
    
    # --- 欧洲 ---
    {'name': '埃菲尔铁塔', 'lat': 48.85, 'lon': 2.29}, # 法国
    {'name': '卢浮宫', 'lat': 48.86, 'lon': 2.33}, # 法国
    {'name': '罗马斗兽场', 'lat': 41.89, 'lon': 12.49}, # 意大利
    {'name': '比萨斜塔', 'lat': 43.72, 'lon': 10.39}, # 意大利
    {'name': '威尼斯广场', 'lat': 45.43, 'lon': 12.33}, # 意大利
    {'name': '帕台农神庙', 'lat': 37.97, 'lon': 23.72}, # 希腊
    {'name': '巨石阵', 'lat': 51.17, 'lon': -1.82}, # 英国
    {'name': '大本钟', 'lat': 51.50, 'lon': -0.12}, # 英国
    {'name': '克里姆林宫', 'lat': 55.75, 'lon': 37.61}, # 俄罗斯
    {'name': '圣瓦西里教堂', 'lat': 55.75, 'lon': 37.62}, # 俄罗斯
    {'name': '新天鹅堡', 'lat': 47.55, 'lon': 10.74}, # 德国
    {'name': '勃兰登堡门', 'lat': 52.51, 'lon': 13.37}, # 德国
    {'name': '圣家堂', 'lat': 41.40, 'lon': 2.17}, # 西班牙
    {'name': '阿尔罕布拉宫', 'lat': 37.17, 'lon': -3.58}, # 西班牙
    {'name': '阿姆斯特丹运河', 'lat': 52.37, 'lon': 4.89}, # 荷兰
    
    # --- 非洲 ---
    {'name': '吉萨金字塔', 'lat': 29.97, 'lon': 31.13}, # 埃及
    {'name': '狮身人面像', 'lat': 29.97, 'lon': 31.13}, # 埃及
    {'name': '卢克索神庙', 'lat': 25.70, 'lon': 32.63}, # 埃及
    {'name': '维多利亚瀑布', 'lat': -17.92, 'lon': 25.85}, # 津巴布韦
    {'name': '桌山', 'lat': -33.96, 'lon': 18.40}, # 南非
    {'name': '杰内大清真寺', 'lat': 13.90, 'lon': -4.55}, # 马里
    
    # --- 美洲 ---
    {'name': '自由女神像', 'lat': 40.68, 'lon': -74.04}, # 美国
    {'name': '大峡谷', 'lat': 36.10, 'lon': -112.11}, # 美国
    {'name': '金门大桥', 'lat': 37.81, 'lon': -122.47}, # 美国
    {'name': '奇琴伊察', 'lat': 20.68, 'lon': -88.56}, # 墨西哥(玛雅)
    {'name': '特奥蒂瓦坎', 'lat': 19.69, 'lon': -98.84}, # 墨西哥
    {'name': '马丘比丘', 'lat': -13.16, 'lon': -72.54}, # 秘鲁
    {'name': '救世基督像', 'lat': -22.95, 'lon': -43.21}, # 巴西
    {'name': '依瓜苏瀑布', 'lat': -25.69, 'lon': -54.43}, # 巴西/阿根廷
    {'name': '复活节岛石像', 'lat': -27.11, 'lon': -109.35}, # 智利
    
    # --- 大洋洲/其他 ---
    {'name': '悉尼歌剧院', 'lat': -33.85, 'lon': 151.21}, # 澳洲
    {'name': '大堡礁', 'lat': -18.28, 'lon': 147.69}, # 澳洲
    {'name': '乌鲁鲁巨石', 'lat': -25.34, 'lon': 131.03}, # 澳洲
    {'name': '霍比特人村', 'lat': -37.87, 'lon': 175.68}, # 新西兰
]
df_landmarks = pd.DataFrame(landmarks_list)

# ==========================================
# 3. 基础底图绘制 (解决俄罗斯问题)
# ==========================================
# 使用 Gapminder 数据作为几何底图
df_geo = px.data.gapminder().query("year == 2007")

# 创建颜色逻辑：如果是Top15国家，用彩色，否则灰色
top15_iso_list = df_top15['iso'].tolist()

def get_color_category(iso):
    if iso == 'CHN': return 'China' # 中国单独一种颜色
    if iso in top15_iso_list: return 'Top 15 Large' # 其他大国
    return 'Others' # 其他国家

df_geo['category'] = df_geo['iso_alpha'].apply(get_color_category)

fig = px.choropleth(
    df_geo,
    locations="iso_alpha",
    color="category",
    hover_name="country", # 确保鼠标悬停显示 Gapminder 自带的准确国名
    projection="orthographic",
    color_discrete_map={
        'China': '#FF0000',       # 中国红
        'Top 15 Large': '#FFD700',# 大国用金色/橙色系
        'Others': '#E0E0E0'       # 其他浅灰
    }
)

# ==========================================
# 4. 图层叠加：Top 15 面积标签
# ==========================================
fig.add_trace(go.Scattergeo(
    lon=df_top15['lon'],
    lat=df_top15['lat'],
    text=df_top15['label'],
    mode='text', # 只显示文字
    textfont=dict(size=10, color='black', family="Arial Black"),
    name='Area Rank'
))

# ==========================================
# 5. 图层叠加：50+ 文化地标
# ==========================================
fig.add_trace(go.Scattergeo(
    lon=df_landmarks['lon'],
    lat=df_landmarks['lat'],
    text=df_landmarks['name'],
    mode='markers+text', # 显示点+字
    marker=dict(
        size=5, 
        color='blue', 
        symbol='circle',
        line=dict(width=0.5, color='white')
    ),
    textposition="top center",
    textfont=dict(size=8, color='darkblue'),
    name='Culture Landmarks'
))

# ==========================================
# 6. 最终美化与输出
# ==========================================
fig.update_geos(
    visible=True,
    showcountries=True,      # 显示国界
    countrycolor="#FFFFFF",  # 国界白色
    showcoastlines=True,
    showland=True,
    landcolor="#E0E0E0",
    showocean=True,
    oceancolor="#B0E2FF",    # 浅蓝海洋
    projection_rotation=dict(lon=105, lat=35, roll=0)
)

fig.update_layout(
    title_text="全球 Top15 面积大国 & 50+ 文化地标分布",
    margin={"r":0,"t":30,"l":0,"b":0},
    showlegend=True
)

fig.show(renderer="browser")

In [7]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np
import math

# ==========================================
# 1. 基础工具函数：距离与方位计算
# ==========================================
def calculate_distance_bearing(lat1, lon1, lat2, lon2):
    # 北京坐标 (固定)
    R = 6371  # 地球半径 (km)
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    
    # Haversine 公式计算距离
    a = math.sin(dphi/2)**2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda/2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    distance = int(R * c)
    
    # 计算方位角
    y = math.sin(dlambda) * math.cos(phi2)
    x = math.cos(phi1) * math.sin(phi2) - math.sin(phi1) * math.cos(phi2) * math.cos(dlambda)
    theta = math.atan2(y, x)
    bearing = (math.degrees(theta) + 360) % 360
    
    # 转换为方位文字
    dirs = ['正北', '东北', '正东', '东南', '正南', '西南', '正西', '西北']
    ix = round(bearing / 45) % 8
    direction = dirs[ix]
    
    return distance, direction

BEIJING_LAT, BEIJING_LON = 39.9042, 116.4074

# ==========================================
# 2. 数据集 A: 全球 Top 15 面积大国 (完全控制颜色)
# ==========================================
top15_data = [
    {'iso': 'RUS', 'name': '俄罗斯', 'rank': 1, 'area': '1709', 'lat': 60, 'lon': 100},
    {'iso': 'CAN', 'name': '加拿大', 'rank': 2, 'area': '998', 'lat': 60, 'lon': -110},
    {'iso': 'CHN', 'name': '中国', 'rank': 3, 'area': '960', 'lat': 35, 'lon': 105},
    {'iso': 'USA', 'name': '美国', 'rank': 4, 'area': '937', 'lat': 40, 'lon': -100},
    {'iso': 'BRA', 'name': '巴西', 'rank': 5, 'area': '851', 'lat': -10, 'lon': -55},
    {'iso': 'AUS', 'name': '澳大利亚', 'rank': 6, 'area': '769', 'lat': -25, 'lon': 135},
    {'iso': 'IND', 'name': '印度', 'rank': 7, 'area': '328', 'lat': 22, 'lon': 79},
    {'iso': 'ARG', 'name': '阿根廷', 'rank': 8, 'area': '278', 'lat': -35, 'lon': -65},
    {'iso': 'KAZ', 'name': '哈萨克斯坦', 'rank': 9, 'area': '272', 'lat': 48, 'lon': 68},
    {'iso': 'DZA', 'name': '阿尔及利亚', 'rank': 10, 'area': '238', 'lat': 28, 'lon': 3},
    {'iso': 'COD', 'name': '刚果(金)', 'rank': 11, 'area': '234', 'lat': -2, 'lon': 23},
    {'iso': 'SAU', 'name': '沙特阿拉伯', 'rank': 12, 'area': '215', 'lat': 24, 'lon': 45},
    {'iso': 'MEX', 'name': '墨西哥', 'rank': 13, 'area': '196', 'lat': 23, 'lon': -102},
    {'iso': 'IDN', 'name': '印度尼西亚', 'rank': 14, 'area': '190', 'lat': -2, 'lon': 120},
    {'iso': 'SDN', 'name': '苏丹', 'rank': 15, 'area': '188', 'lat': 16, 'lon': 30},
]
df_top15 = pd.DataFrame(top15_data)
df_top15['label'] = df_top15.apply(lambda x: f"#{x['rank']} {x['name']}", axis=1)

# 为 Top 15 设置颜色列
def get_color(iso):
    if iso == 'CHN': return '#FF0000' # 中国红
    if iso == 'RUS': return '#0055A4' # 俄罗斯蓝
    return '#FFD700' # 其他大国金色

df_top15['color_code'] = df_top15['iso'].apply(get_color)

# ==========================================
# 3. 数据集 B: 50+ 文化地标 (用于计算表格和地图)
# ==========================================
landmarks_list = [
    # 亚洲
    {'name': '长城', 'country': '中国', 'lat': 40.43, 'lon': 116.57},
    {'name': '故宫', 'country': '中国', 'lat': 39.91, 'lon': 116.39},
    {'name': '秦始皇兵马俑', 'country': '中国', 'lat': 34.38, 'lon': 109.27},
    {'name': '布达拉宫', 'country': '中国', 'lat': 29.65, 'lon': 91.11},
    {'name': '泰姬陵', 'country': '印度', 'lat': 27.17, 'lon': 78.04},
    {'name': '吴哥窟', 'country': '柬埔寨', 'lat': 13.41, 'lon': 103.86},
    {'name': '富士山', 'country': '日本', 'lat': 35.36, 'lon': 138.72},
    {'name': '京都金阁寺', 'country': '日本', 'lat': 35.03, 'lon': 135.72},
    {'name': '佩特拉古城', 'country': '约旦', 'lat': 30.32, 'lon': 35.44},
    {'name': '哈利法塔', 'country': '阿联酋', 'lat': 25.19, 'lon': 55.27},
    {'name': '婆罗浮屠', 'country': '印度尼西亚', 'lat': -7.60, 'lon': 110.20},
    {'name': '仰光大金塔', 'country': '缅甸', 'lat': 16.79, 'lon': 96.14},
    {'name': '麦加禁寺', 'country': '沙特阿拉伯', 'lat': 21.42, 'lon': 39.82},
    {'name': '波斯波利斯', 'country': '伊朗', 'lat': 29.93, 'lon': 52.89},
    # 欧洲
    {'name': '埃菲尔铁塔', 'country': '法国', 'lat': 48.85, 'lon': 2.29},
    {'name': '卢浮宫', 'country': '法国', 'lat': 48.86, 'lon': 2.33},
    {'name': '罗马斗兽场', 'country': '意大利', 'lat': 41.89, 'lon': 12.49},
    {'name': '比萨斜塔', 'country': '意大利', 'lat': 43.72, 'lon': 10.39},
    {'name': '威尼斯广场', 'country': '意大利', 'lat': 45.43, 'lon': 12.33},
    {'name': '帕台农神庙', 'country': '希腊', 'lat': 37.97, 'lon': 23.72},
    {'name': '巨石阵', 'country': '英国', 'lat': 51.17, 'lon': -1.82},
    {'name': '大本钟', 'country': '英国', 'lat': 51.50, 'lon': -0.12},
    {'name': '克里姆林宫', 'country': '俄罗斯', 'lat': 55.75, 'lon': 37.61},
    {'name': '圣瓦西里大教堂', 'country': '俄罗斯', 'lat': 55.75, 'lon': 37.62},
    {'name': '新天鹅堡', 'country': '德国', 'lat': 47.55, 'lon': 10.74},
    {'name': '勃兰登堡门', 'country': '德国', 'lat': 52.51, 'lon': 13.37},
    {'name': '圣家堂', 'country': '西班牙', 'lat': 41.40, 'lon': 2.17},
    {'name': '阿尔罕布拉宫', 'country': '西班牙', 'lat': 37.17, 'lon': -3.58},
    {'name': '阿姆斯特丹运河', 'country': '荷兰', 'lat': 52.37, 'lon': 4.89},
    # 非洲
    {'name': '吉萨金字塔', 'country': '埃及', 'lat': 29.97, 'lon': 31.13},
    {'name': '狮身人面像', 'country': '埃及', 'lat': 29.97, 'lon': 31.13},
    {'name': '卢克索神庙', 'country': '埃及', 'lat': 25.70, 'lon': 32.63},
    {'name': '维多利亚瀑布', 'country': '津巴布韦/赞比亚', 'lat': -17.92, 'lon': 25.85},
    {'name': '桌山', 'country': '南非', 'lat': -33.96, 'lon': 18.40},
    {'name': '杰内大清真寺', 'country': '马里', 'lat': 13.90, 'lon': -4.55},
    # 美洲
    {'name': '自由女神像', 'country': '美国', 'lat': 40.68, 'lon': -74.04},
    {'name': '大峡谷', 'country': '美国', 'lat': 36.10, 'lon': -112.11},
    {'name': '金门大桥', 'country': '美国', 'lat': 37.81, 'lon': -122.47},
    {'name': '奇琴伊察(玛雅)', 'country': '墨西哥', 'lat': 20.68, 'lon': -88.56},
    {'name': '特奥蒂瓦坎', 'country': '墨西哥', 'lat': 19.69, 'lon': -98.84},
    {'name': '马丘比丘', 'country': '秘鲁', 'lat': -13.16, 'lon': -72.54},
    {'name': '救世基督像', 'country': '巴西', 'lat': -22.95, 'lon': -43.21},
    {'name': '伊瓜苏瀑布', 'country': '巴西/阿根廷', 'lat': -25.69, 'lon': -54.43},
    {'name': '复活节岛石像', 'country': '智利', 'lat': -27.11, 'lon': -109.35},
    # 大洋洲
    {'name': '悉尼歌剧院', 'country': '澳大利亚', 'lat': -33.85, 'lon': 151.21},
    {'name': '大堡礁', 'country': '澳大利亚', 'lat': -18.28, 'lon': 147.69},
    {'name': '乌鲁鲁巨石', 'country': '澳大利亚', 'lat': -25.34, 'lon': 131.03},
    {'name': '霍比特人村', 'country': '新西兰', 'lat': -37.87, 'lon': 175.68},
]

# 计算距离和方位
rows = []
for lm in landmarks_list:
    dist, bear = calculate_distance_bearing(BEIJING_LAT, BEIJING_LON, lm['lat'], lm['lon'])
    lm['distance_km'] = dist
    lm['bearing_text'] = bear
    rows.append(lm)

df_landmarks = pd.DataFrame(rows)
df_landmarks = df_landmarks.sort_values(by='distance_km') # 按距离北京远近排序

# ==========================================
# 4. 数据集 C: 赤道/热带旅游胜地 (新增要求)
# ==========================================
equator_spots = [
    {'name': '马尔代夫', 'lat': 3.20, 'lon': 73.22},
    {'name': '新加坡', 'lat': 1.35, 'lon': 103.81},
    {'name': '巴厘岛', 'lat': -8.40, 'lon': 115.18},
    {'name': '塞舌尔', 'lat': -4.67, 'lon': 55.49},
    {'name': '加拉帕戈斯群岛', 'lat': -0.95, 'lon': -90.96},
    {'name': '亚马逊雨林', 'lat': -3.11, 'lon': -60.02},
    {'name': '肯尼亚(马赛马拉)', 'lat': -1.29, 'lon': 35.14},
    {'name': '基多(赤道之城)', 'lat': -0.18, 'lon': -78.46},
    {'name': '大溪地', 'lat': -17.65, 'lon': -149.40}
]
df_equator = pd.DataFrame(equator_spots)


# ==========================================
# 5. 绘图 1: 3D 地球仪 (Globe)
# ==========================================
# 直接使用 df_top15 绘制底图，确保俄罗斯一定有颜色
fig_globe = px.choropleth(
    df_top15,
    locations="iso",
    color="iso",  # 使用 ISO 作为颜色分类的依据
    hover_name="name",
    projection="orthographic",
    color_discrete_map={row['iso']: row['color_code'] for index, row in df_top15.iterrows()}
)

# A. 添加 Top 15 标签
fig_globe.add_trace(go.Scattergeo(
    lon=df_top15['lon'], lat=df_top15['lat'],
    text=df_top15['label'],
    mode='text',
    textfont=dict(size=10, color='black', family="Arial Black"),
    name='Area Rank'
))

# B. 添加赤道旅游胜地 (新增) - 绿色醒目圆点
fig_globe.add_trace(go.Scattergeo(
    lon=df_equator['lon'], lat=df_equator['lat'],
    text=df_equator['name'],
    mode='markers+text',
    marker=dict(size=8, color='#00FF7F', symbol='circle', line=dict(width=1, color='white')), # 春绿色
    textposition="bottom center",
    textfont=dict(size=9, color='#00FF7F'),
    name='Equatorial Holiday Spots'
))

# C. 添加文化地标 - 蓝色圆点
fig_globe.add_trace(go.Scattergeo(
    lon=df_landmarks['lon'], lat=df_landmarks['lat'],
    text=df_landmarks['name'],
    mode='markers', # 仅显示点，鼠标悬停看名字，避免地图太乱
    marker=dict(size=5, color='blue', symbol='circle'),
    hovertext=df_landmarks['name'] + ' (' + df_landmarks['country'] + ')',
    name='Culture Landmarks'
))

# D. 全局美化
fig_globe.update_geos(
    visible=True,
    showcountries=True,
    countrycolor="#CCCCCC", # 其他国家的边界线颜色
    showland=True,
    landcolor="#F0F0F0",    # 其他国家默认显示浅灰色 (解决俄罗斯之前的底色问题)
    showocean=True,
    oceancolor="#B0E2FF",
    projection_rotation=dict(lon=105, lat=20, roll=0)
)
fig_globe.update_layout(
    title_text="3D Globe: Top15 Countries & Equatorial Spots",
    margin={"r":0,"t":40,"l":0,"b":0},
    showlegend=True
)

# ==========================================
# 6. 绘图 2: 详细数据表格 (Table)
# ==========================================
fig_table = go.Figure(data=[go.Table(
    header=dict(
        values=['<b>文化地标名称</b>', '<b>所在国家</b>', '<b>距离北京 (km)</b>', '<b>相对北京方位</b>', '<b>经纬度</b>'],
        fill_color='paleturquoise',
        align='left',
        font=dict(size=12, color='black')
    ),
    cells=dict(
        values=[
            df_landmarks['name'], 
            df_landmarks['country'], 
            df_landmarks['distance_km'], 
            df_landmarks['bearing_text'],
            df_landmarks.apply(lambda x: f"{x['lat']:.1f}, {x['lon']:.1f}", axis=1)
        ],
        fill_color='lavender',
        align='left',
        font=dict(size=11),
        height=25
    )
)])

fig_table.update_layout(
    title_text="文化设施详情表 (按距京距离排序)",
    margin={"r":20,"t":50,"l":20,"b":20},
)

# ==========================================
# 7. 打开两个独立窗口
# ==========================================
fig_globe.show(renderer="browser") # 窗口1：地球仪
fig_table.show(renderer="browser") # 窗口2：表格

In [9]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np
import math

# ==========================================
# 1. 基础工具：生成标志线 & 计算距离
# ==========================================
def generate_lat_line(lat_value):
    """生成一条指定纬度的线，环绕地球一圈"""
    lons = np.linspace(-180, 180, 360)
    lats = [lat_value] * 360
    return lons, lats

def calculate_distance_bearing(lat1, lon1, lat2, lon2):
    """计算两点间距离和方位"""
    R = 6371  # 地球半径 km
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda/2)**2
    distance = int(R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a)))
    
    y = math.sin(dlambda) * math.cos(phi2)
    x = math.cos(phi1) * math.sin(phi2) - math.sin(phi1) * math.cos(phi2) * math.cos(dlambda)
    bearing = (math.degrees(math.atan2(y, x)) + 360) % 360
    
    dirs = ['正北', '东北', '正东', '东南', '正南', '西南', '正西', '西北']
    direction = dirs[round(bearing / 45) % 8]
    return distance, direction

# ==========================================
# 2. 数据准备：Top 15、全量国家、地标
# ==========================================
# Top 15 数据 (用于标签和特定上色)
top15_data = [
    {'iso': 'RUS', 'name': '俄罗斯', 'rank': 1, 'area': '1709', 'lat': 60, 'lon': 100},
    {'iso': 'CAN', 'name': '加拿大', 'rank': 2, 'area': '998', 'lat': 60, 'lon': -110},
    {'iso': 'CHN', 'name': '中国', 'rank': 3, 'area': '960', 'lat': 35, 'lon': 105},
    {'iso': 'USA', 'name': '美国', 'rank': 4, 'area': '937', 'lat': 40, 'lon': -100},
    {'iso': 'BRA', 'name': '巴西', 'rank': 5, 'area': '851', 'lat': -10, 'lon': -55},
    {'iso': 'AUS', 'name': '澳大利亚', 'rank': 6, 'area': '769', 'lat': -25, 'lon': 135},
    {'iso': 'IND', 'name': '印度', 'rank': 7, 'area': '328', 'lat': 22, 'lon': 79},
    {'iso': 'ARG', 'name': '阿根廷', 'rank': 8, 'area': '278', 'lat': -35, 'lon': -65},
    {'iso': 'KAZ', 'name': '哈萨克斯坦', 'rank': 9, 'area': '272', 'lat': 48, 'lon': 68},
    {'iso': 'DZA', 'name': '阿尔及利亚', 'rank': 10, 'area': '238', 'lat': 28, 'lon': 3},
    {'iso': 'COD', 'name': '刚果(金)', 'rank': 11, 'area': '234', 'lat': -2, 'lon': 23},
    {'iso': 'SAU', 'name': '沙特阿拉伯', 'rank': 12, 'area': '215', 'lat': 24, 'lon': 45},
    {'iso': 'MEX', 'name': '墨西哥', 'rank': 13, 'area': '196', 'lat': 23, 'lon': -102},
    {'iso': 'IDN', 'name': '印度尼西亚', 'rank': 14, 'area': '190', 'lat': -2, 'lon': 120},
    {'iso': 'SDN', 'name': '苏丹', 'rank': 15, 'area': '188', 'lat': 16, 'lon': 30},
]
df_top15 = pd.DataFrame(top15_data)
top15_iso_list = df_top15['iso'].tolist()

# 核心修正：加载完整的世界国家数据，确保每个国家都有交互
df_world = px.data.gapminder().query("year == 2007")

# 定义颜色分类逻辑
def assign_color_group(iso):
    if iso == 'CHN': return 'China (Red)'          # 中国
    if iso == 'RUS': return 'Russia (Blue)'        # 俄罗斯
    if iso in top15_iso_list: return 'Top 15 (Gold)' # 其他大国
    return 'Rest of World'                         # 其他所有国家

df_world['color_group'] = df_world['iso_alpha'].apply(assign_color_group)

# 地标数据
landmarks_list = [
    # 亚洲
    {'name': '长城', 'country': '中国', 'lat': 40.43, 'lon': 116.57},
    {'name': '故宫', 'country': '中国', 'lat': 39.91, 'lon': 116.39},
    {'name': '秦始皇兵马俑', 'country': '中国', 'lat': 34.38, 'lon': 109.27},
    {'name': '布达拉宫', 'country': '中国', 'lat': 29.65, 'lon': 91.11},
    {'name': '泰姬陵', 'country': '印度', 'lat': 27.17, 'lon': 78.04},
    {'name': '吴哥窟', 'country': '柬埔寨', 'lat': 13.41, 'lon': 103.86},
    {'name': '富士山', 'country': '日本', 'lat': 35.36, 'lon': 138.72},
    {'name': '京都金阁寺', 'country': '日本', 'lat': 35.03, 'lon': 135.72},
    {'name': '佩特拉古城', 'country': '约旦', 'lat': 30.32, 'lon': 35.44},
    {'name': '哈利法塔', 'country': '阿联酋', 'lat': 25.19, 'lon': 55.27},
    {'name': '婆罗浮屠', 'country': '印度尼西亚', 'lat': -7.60, 'lon': 110.20},
    {'name': '仰光大金塔', 'country': '缅甸', 'lat': 16.79, 'lon': 96.14},
    {'name': '麦加禁寺', 'country': '沙特阿拉伯', 'lat': 21.42, 'lon': 39.82},
    {'name': '波斯波利斯', 'country': '伊朗', 'lat': 29.93, 'lon': 52.89},
    # 欧洲
    {'name': '埃菲尔铁塔', 'country': '法国', 'lat': 48.85, 'lon': 2.29},
    {'name': '卢浮宫', 'country': '法国', 'lat': 48.86, 'lon': 2.33},
    {'name': '罗马斗兽场', 'country': '意大利', 'lat': 41.89, 'lon': 12.49},
    {'name': '比萨斜塔', 'country': '意大利', 'lat': 43.72, 'lon': 10.39},
    {'name': '威尼斯广场', 'country': '意大利', 'lat': 45.43, 'lon': 12.33},
    {'name': '帕台农神庙', 'country': '希腊', 'lat': 37.97, 'lon': 23.72},
    {'name': '巨石阵', 'country': '英国', 'lat': 51.17, 'lon': -1.82},
    {'name': '大本钟', 'country': '英国', 'lat': 51.50, 'lon': -0.12},
    {'name': '克里姆林宫', 'country': '俄罗斯', 'lat': 55.75, 'lon': 37.61},
    {'name': '圣瓦西里大教堂', 'country': '俄罗斯', 'lat': 55.75, 'lon': 37.62},
    {'name': '新天鹅堡', 'country': '德国', 'lat': 47.55, 'lon': 10.74},
    {'name': '勃兰登堡门', 'country': '德国', 'lat': 52.51, 'lon': 13.37},
    {'name': '圣家堂', 'country': '西班牙', 'lat': 41.40, 'lon': 2.17},
    {'name': '阿尔罕布拉宫', 'country': '西班牙', 'lat': 37.17, 'lon': -3.58},
    {'name': '阿姆斯特丹运河', 'country': '荷兰', 'lat': 52.37, 'lon': 4.89},
    # 非洲
    {'name': '吉萨金字塔', 'country': '埃及', 'lat': 29.97, 'lon': 31.13},
    {'name': '狮身人面像', 'country': '埃及', 'lat': 29.97, 'lon': 31.13},
    {'name': '卢克索神庙', 'country': '埃及', 'lat': 25.70, 'lon': 32.63},
    {'name': '维多利亚瀑布', 'country': '津巴布韦/赞比亚', 'lat': -17.92, 'lon': 25.85},
    {'name': '桌山', 'country': '南非', 'lat': -33.96, 'lon': 18.40},
    {'name': '杰内大清真寺', 'country': '马里', 'lat': 13.90, 'lon': -4.55},
    # 美洲
    {'name': '自由女神像', 'country': '美国', 'lat': 40.68, 'lon': -74.04},
    {'name': '大峡谷', 'country': '美国', 'lat': 36.10, 'lon': -112.11},
    {'name': '金门大桥', 'country': '美国', 'lat': 37.81, 'lon': -122.47},
    {'name': '奇琴伊察(玛雅)', 'country': '墨西哥', 'lat': 20.68, 'lon': -88.56},
    {'name': '特奥蒂瓦坎', 'country': '墨西哥', 'lat': 19.69, 'lon': -98.84},
    {'name': '马丘比丘', 'country': '秘鲁', 'lat': -13.16, 'lon': -72.54},
    {'name': '救世基督像', 'country': '巴西', 'lat': -22.95, 'lon': -43.21},
    {'name': '伊瓜苏瀑布', 'country': '巴西/阿根廷', 'lat': -25.69, 'lon': -54.43},
    {'name': '复活节岛石像', 'country': '智利', 'lat': -27.11, 'lon': -109.35},
    # 大洋洲
    {'name': '悉尼歌剧院', 'country': '澳大利亚', 'lat': -33.85, 'lon': 151.21},
    {'name': '大堡礁', 'country': '澳大利亚', 'lat': -18.28, 'lon': 147.69},
    {'name': '乌鲁鲁巨石', 'country': '澳大利亚', 'lat': -25.34, 'lon': 131.03},
    {'name': '霍比特人村', 'country': '新西兰', 'lat': -37.87, 'lon': 175.68},
]
# 为了演示表格效果，简单补充几个
for i in range(5): landmarks_list.append({'name': f'示例地标{i}', 'country': '示例国', 'lat': 0, 'lon': i*10})

# 计算距离
BEIJING_LAT, BEIJING_LON = 39.9042, 116.4074
for lm in landmarks_list:
    d, b = calculate_distance_bearing(BEIJING_LAT, BEIJING_LON, lm['lat'], lm['lon'])
    lm['distance_km'], lm['bearing'] = d, b
df_landmarks = pd.DataFrame(landmarks_list).sort_values('distance_km')

# 赤道景点
equator_spots = [
    {'name': '马尔代夫', 'lat': 3.20, 'lon': 73.22},
    {'name': '新加坡', 'lat': 1.35, 'lon': 103.81},
    {'name': '巴厘岛', 'lat': -8.40, 'lon': 115.18},
    {'name': '亚马逊雨林', 'lat': -3.11, 'lon': -60.02},
    {'name': '基多(赤道碑)', 'lat': -0.18, 'lon': -78.46}
]
df_equator = pd.DataFrame(equator_spots)

# ==========================================
# 3. 绘图：3D 地球仪 (包含所有图层)
# ==========================================

# 3.1 基础底图 (使用 df_world 确保所有国家可点击)
fig = px.choropleth(
    df_world,
    locations="iso_alpha",
    color="color_group",
    hover_name="country", # 这里解决了你的问题：其他国家现在可以显示名字了
    projection="orthographic",
    color_discrete_map={
        'China (Red)': '#FF0000',
        'Russia (Blue)': '#0055A4',
        'Top 15 (Gold)': '#FFD700',
        'Rest of World': '#DDDDDD' # 其他国家浅灰，但存在实体
    }
)

# 3.2 绘制标志线 (赤道、回归线)
# 赤道
lons_eq, lats_eq = generate_lat_line(0)
fig.add_trace(go.Scattergeo(
    lon=lons_eq, lat=lats_eq, mode='lines',
    line=dict(width=3, color='red', dash='solid'),
    name='Equator (赤道)', hoverinfo='text', text='赤道 0°'
))

# 北回归线
lons_can, lats_can = generate_lat_line(23.4368)
fig.add_trace(go.Scattergeo(
    lon=lons_can, lat=lats_can, mode='lines',
    line=dict(width=2, color='orange', dash='dash'),
    name='Tropic of Cancer (北回归线)', hoverinfo='text', text='北回归线 23.5°N'
))

# 南回归线
lons_cap, lats_cap = generate_lat_line(-23.4368)
fig.add_trace(go.Scattergeo(
    lon=lons_cap, lat=lats_cap, mode='lines',
    line=dict(width=2, color='orange', dash='dash'),
    name='Tropic of Capricorn (南回归线)', hoverinfo='text', text='南回归线 23.5°S'
))

# 3.3 添加 Top 15 面积标签
fig.add_trace(go.Scattergeo(
    lon=df_top15['lon'], lat=df_top15['lat'],
    text=df_top15.apply(lambda x: f"#{x['rank']} {x['name']}", axis=1),
    mode='text',
    textfont=dict(size=10, color='black', family="Arial Black"),
    name='Rank Labels'
))

# 3.4 添加赤道旅游景点 (绿色点)
fig.add_trace(go.Scattergeo(
    lon=df_equator['lon'], lat=df_equator['lat'],
    text=df_equator['name'],
    mode='markers+text',
    marker=dict(size=8, color='#00FF7F', line=dict(width=1, color='black')),
    textposition="bottom center",
    name='Equator Spots'
))

# 3.5 添加文化地标 (蓝色点)
fig.add_trace(go.Scattergeo(
    lon=df_landmarks['lon'], lat=df_landmarks['lat'],
    text=df_landmarks['name'],
    mode='markers',
    marker=dict(size=5, color='blue'),
    hovertext=df_landmarks['name'],
    name='Culture Landmarks'
))

# 3.6 布局设置
fig.update_geos(
    visible=True,
    showcountries=True, countrycolor="white", # 开启国界线
    showland=True, landcolor="#DDDDDD",
    showocean=True, oceancolor="#B0E2FF",
    projection_rotation=dict(lon=105, lat=20, roll=0)
)
fig.update_layout(
    title_text="Global Visualization: Countries, Landmarks & Reference Lines",
    margin={"r":0,"t":30,"l":0,"b":0},
    legend=dict(x=0, y=1)
)

# ==========================================
# 4. 绘图：数据表格 (Table)
# ==========================================
fig_table = go.Figure(data=[go.Table(
    header=dict(
        values=['<b>地标名称</b>', '<b>国家</b>', '<b>距北京 (km)</b>', '<b>方位</b>', '<b>经纬度</b>'],
        fill_color='paleturquoise', align='center'
    ),
    cells=dict(
        values=[
            df_landmarks['name'], df_landmarks['country'], 
            df_landmarks['distance_km'], df_landmarks['bearing'],
            df_landmarks.apply(lambda x: f"{x['lat']:.1f}, {x['lon']:.1f}", axis=1)
        ],
        fill_color='lavender', align='center'
    )
)])
fig_table.update_layout(title="文化地标详情表 (按距离排序)", margin={"t":50,"b":10})

# ==========================================
# 5. 显示
# ==========================================
fig.show(renderer="browser")
fig_table.show(renderer="browser")

In [11]:
# 打开一个新文件写入
with open("JZL_Full_Report.html", 'w', encoding='utf-8') as f:
    f.write('<h1>全球地理可视化分析报告 (Made by JZL)</h1>')
    f.write(fig.to_html(full_html=False, include_plotlyjs='cdn')) # 写入地球仪
    f.write('<br><hr><br>') # 加个换行和分割线
    f.write(fig_table.to_html(full_html=False, include_plotlyjs='cdn')) # 写入表格

print("合并版网页已生成：JZL_Full_Report.html")

合并版网页已生成：JZL_Full_Report.html


In [12]:
# 确保你已经运行了之前的绘图代码，生成了 fig 和 fig_table

# 1. 我们把地球仪保存为 index.html
# 建议把所有内容合并到一个网页，方便分享
with open("index.html", 'w', encoding='utf-8') as f:
    f.write('<html><head><title>JZL Earth Visualization</title></head><body>')
    f.write('<h1 style="text-align:center; font-family:Arial;">JZL 全球地理可视化系统 (Master Thesis Demo)</h1>')
    f.write(fig.to_html(full_html=False, include_plotlyjs='cdn')) # 写入地球仪
    f.write('<br><hr><br>') 
    f.write(fig_table.to_html(full_html=False, include_plotlyjs='cdn')) # 写入表格
    f.write('</body></html>')

print("文件已在本地生成：index.html")
print("下一步：请查看下方的上传教程来获取 URL。")

文件已在本地生成：index.html
下一步：请查看下方的上传教程来获取 URL。
